# DP2 — DiaObject / DiaSource catalog query over a DDF via Butler

**Author:** dagoret  
**Date:** 2026-03-13  

## Purpose

This notebook retrieves **DiaObject**, **DiaSource**, and **ForcedSourceOnDiaObject**
catalogs for a user-selected Deep Drilling Field (DDF) using the **Butler Gen3 only**
(no TAP service, which is not yet available for DP2 at USDF).

The strategy is:
1. Identify the tracts from `lsst_cells_v2` that cover the selected DDF.
2. Enumerate available `dia_object` dataset references for those tracts via
   `butler.registry.queryDatasets`.
3. Load each parquet file with `butler.get`, apply a spatial cone cut and a
   `nDiaSources >= MIN_NSOURCES` filter.
4. Load the matching `dia_source` and forced-source datasets to build per-object
   light curves.
5. Provide a **coordinate-based cross-match** utility for Fink broker alerts:
   given an (RA, Dec) from Fink, find the closest DRP DiaObject by angular
   separation and retrieve its forced-photometry light curve.

## Why coordinate cross-match instead of diaObjectId?

The **Rubin Alert Production** pipeline (which feeds brokers such as Fink,
ALeRCE, Lasair …) and the **DRP** are **independent pipelines**.  They assign
their own `diaObjectId` values to the same physical source; these IDs are
**not cross-compatible**.  The only robust identification strategy is a
sky-coordinate cross-match using a matching radius of a few arcseconds.

## Data products

| Butler dataset type | Dimensions | Content |
|---|---|---|
| `dia_object` | `{skymap, tract}` | Summary statistics per transient/variable object |
| `dia_source` | `{skymap, tract}` | Individual ≥5σ difference-image detections |
| `forced_diff_exp_source_on_dia_object` | `{skymap, tract}` | Forced PSF photometry at every epoch |

> **Note:** dataset type names were verified via
> `butler.registry.getDatasetType('dia_object').dimensions`
> in notebook `2026-03-07_AccessToDP2.ipynb`.  
> If a dataset type is absent from the collection, a clear error message is printed.

## Reference notebooks

- `2026-03-07_AccessToDP2.ipynb` — Butler setup, dataset type exploration  
- `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb` — SkyMap tract lookup

---
## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Astropy
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack
import astropy.units as u

# LSST Science Pipelines
import lsst.daf.butler as dafButler
from lsst.daf.butler import Butler
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

print(f"Matplotlib : {matplotlib.__version__}")
print(f"Seaborn    : {sns.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

---
## 1. User Configuration

**Edit this cell only** to select the DDF and tune the query parameters.

In [ ]:
# =========================================================================
# USER PARAMETERS — edit here
# =========================================================================

# --- Target DDF ---
# Choose one key from DDF_COORDS below.
# Available: "COSMOS", "ECDFS", "ELAIS-S1", "XMM-LSS",
#            "EDFS-a", "EDFS-b", "EDFS", "M49"
SELECTED_DDF = "ECDFS"

# --- Search cone radius (degrees) around the DDF center ---
# Use ~1.8° to cover all tracts that overlap the DDF.
CONE_RADIUS_DEG = 1.8

# --- Minimum number of DiaSources per DiaObject ---
# Objects with fewer detections are likely single-epoch events or artifacts.
MIN_NSOURCES = 50

# --- Butler repository and collection ---
REPO        = "dp2_prep"
COLLECTION  = "LSSTCam/runs/DRP/v30_0_4/DM-54249"
SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT  = "LSSTCam"

# --- Tract search radius for SkyMap lookup (degrees) ---
TRACT_SEARCH_RADIUS_DEG = 1.8

# =========================================================================
# DDF coordinate dictionary  (RA, Dec in degrees, ICRS J2000)
# Source: 2026-03-11_DP2_SurveyPropertyMaps_AllBands_AllDDF.ipynb
# =========================================================================
DDF_COORDS = {
    "COSMOS"   : (150.119, +2.206),
    "ECDFS"    : ( 53.125, -28.100),   # Extended Chandra Deep Field South
    "ELAIS-S1" : (  9.450, -44.000),
    "XMM-LSS"  : ( 35.708,  -4.750),
    "EDFS-a"   : ( 58.900, -49.315),   # Euclid Deep Field South (a)
    "EDFS-b"   : ( 63.600, -47.600),   # Euclid Deep Field South (b)
    "EDFS"     : ( 61.240, -48.423),
    "M49"      : (187.400,  +8.000),
}

# Resolve the selected DDF
if SELECTED_DDF not in DDF_COORDS:
    raise ValueError(
        f"Unknown DDF: '{SELECTED_DDF}'. "
        f"Available: {list(DDF_COORDS.keys())}"
    )
RA_CENTER, DEC_CENTER = DDF_COORDS[SELECTED_DDF]

print(f"Selected DDF    : {SELECTED_DDF}")
print(f"Center RA       : {RA_CENTER:.4f}°")
print(f"Center Dec      : {DEC_CENTER:+.4f}°")
print(f"Cone radius     : {CONE_RADIUS_DEG}°")
print(f"Min nDiaSources : {MIN_NSOURCES}")
print(f"Butler repo     : {REPO}")
print(f"Collection      : {COLLECTION}")
print(f"SkyMap          : {SKYMAP_NAME}")

---
## 2. Butler and SkyMap Initialisation

In [ ]:
# -------------------------------------------------------------------------
# Instantiate the Butler
# -------------------------------------------------------------------------
butler   = Butler(REPO, collections=COLLECTION)
registry = butler.registry

print(f"Butler     : {REPO}")
print(f"Collection : {COLLECTION}")

In [ ]:
# -------------------------------------------------------------------------
# Load the SkyMap
# The skyMap dataset may live in a parent CHAINED collection; try both.
# -------------------------------------------------------------------------
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)

print(f"SkyMap loaded   : {SKYMAP_NAME}")
print(f"Number of tracts: {len(skymap)}")

---
## 3. Introspect Available Dataset Types

List all science-relevant dataset types present in the collection, then inspect
the **dimension requirements** of `dia_object`, `dia_source`, and forced-source
products.  This determines which keys must appear in `dataId` dicts.

In [ ]:
# -------------------------------------------------------------------------
# List all science dataset types present in the collection
# (skip pipeline bookkeeping: _config, _log, _metadata, Plot, Metric)
# -------------------------------------------------------------------------
SKIP_SUFFIXES = ('_config', '_log', '_metadata', '_resource_usage',
                 'Plot', 'Metric', 'metric')

science_types = []
for dt in registry.queryDatasetTypes():
    if any(s in dt.name for s in SKIP_SUFFIXES):
        continue
    if registry.queryDatasets(dt, collections=COLLECTION).any(
            execute=False, exact=False):
        science_types.append(dt.name)

science_types.sort()
print(f"Science dataset types in collection ({len(science_types)}):")
for n in science_types:
    print(f"  {n}")

In [ ]:
# -------------------------------------------------------------------------
# Inspect the dimension requirements of the DIA dataset types.
# The dimensions determine which keys must appear in the dataId dict.
# -------------------------------------------------------------------------
DIA_CANDIDATES = [
    "dia_object",
    "dia_source",
    "forced_diff_exp_source_on_dia_object",
    "dia_object_forced_source",
    "forcedSourceOnDiaObject",
]

for dt_name in DIA_CANDIDATES:
    try:
        dt = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(
            dt_name, collections=COLLECTION
        ).any(execute=False, exact=False)
        print(f"{dt_name:50s}  dims={list(dt.dimensions.names)}  in_collection={in_col}")
    except Exception as exc:
        print(f"{dt_name:50s}  NOT FOUND ({exc.__class__.__name__})")

In [ ]:
# -------------------------------------------------------------------------
# Select the forced-source dataset type that is actually present
# in the collection (name varies between DRP processing runs).
# -------------------------------------------------------------------------
FORCED_ON_DIA_TYPE = None
for candidate in [
    "forced_diff_exp_source_on_dia_object",
    "dia_object_forced_source",
    "forcedSourceOnDiaObject",
]:
    try:
        registry.getDatasetType(candidate)
        if registry.queryDatasets(
            candidate, collections=COLLECTION
        ).any(execute=False, exact=False):
            FORCED_ON_DIA_TYPE = candidate
            break
    except Exception:
        pass

print(f"Forced-source dataset type selected: {FORCED_ON_DIA_TYPE}")

---
## 4. Find Tracts Covering the Selected DDF

Reuses the grid-sampling approach from
`2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb`.

In [ ]:
# -------------------------------------------------------------------------
# Tract lookup via grid-sampling of the search disk
# -------------------------------------------------------------------------
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    """
    Return deduplicated list of TractInfo whose footprint overlaps
    the disk (ra_deg, dec_deg, radius_deg).

    Uses only skymap.findTract(SpherePoint) — stable across stack versions.
    """
    cos_dec   = max(np.cos(np.deg2rad(dec_deg)), 0.01)  # avoid division by zero at poles
    step      = 0.35                                     # grid step in degrees
    found_ids = set()

    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s  = ra_deg  + dra / cos_dec   # projection correction in RA
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                ti = skymap.findTract(sp)
                found_ids.add(ti.tract_id)
            except Exception:
                pass

    return [skymap[tid] for tid in sorted(found_ids)]


ddf_tract_infos = find_tracts_for_coord(
    skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG
)
tract_ids = [t.tract_id for t in ddf_tract_infos]

print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

---
## 5. Load DiaObject Catalog via Butler

For each tract:
1. Query `butler.registry.queryDatasets('dia_object', ...)` to get references.
2. Load each reference with `butler.get()`.
3. Concatenate across tracts.
4. Apply a spatial cone cut with `astropy.coordinates.SkyCoord.separation()`.
5. Filter on `nDiaSources >= MIN_NSOURCES`.

In [ ]:
# -------------------------------------------------------------------------
# Helper: convert whatever butler.get returns to a pandas DataFrame.
# Handles ArrowAstropy, astropy Table, and plain DataFrame.
# -------------------------------------------------------------------------
def to_dataframe(obj):
    """Convert a Butler catalog object to a pandas DataFrame."""
    if isinstance(obj, pd.DataFrame):
        return obj
    if hasattr(obj, 'to_pandas'):
        return obj.to_pandas()
    if hasattr(obj, 'to_table'):
        return obj.to_table().to_pandas()
    try:
        if isinstance(obj, Table):
            return obj.to_pandas()
    except Exception:
        pass
    raise TypeError(f"Cannot convert {type(obj)} to pandas DataFrame.")

In [ ]:
# -------------------------------------------------------------------------
# Query dia_object references for all tracts that cover the DDF
# -------------------------------------------------------------------------
print("Querying dia_object references ...")

dia_obj_refs = []
for tract_id in tract_ids:
    dataId = {"skymap": SKYMAP_NAME, "tract": tract_id}
    refs = list(
        registry.queryDatasets(
            "dia_object",
            collections=COLLECTION,
            dataId=dataId,
        )
    )
    print(f"  Tract {tract_id:6d} — {len(refs)} dia_object reference(s)")
    dia_obj_refs.extend(refs)

print(f"\nTotal dia_object references: {len(dia_obj_refs)}")

In [ ]:
# -------------------------------------------------------------------------
# Load all dia_object references and concatenate into a single DataFrame
# -------------------------------------------------------------------------
dia_obj_frames = []

for ref in dia_obj_refs:
    tract_id = ref.dataId['tract']
    try:
        obj    = butler.get(ref)
        df_tmp = to_dataframe(obj)
        df_tmp['_tract'] = tract_id   # store provenance
        dia_obj_frames.append(df_tmp)
        print(f"  Tract {tract_id:6d} — {len(df_tmp):,} DiaObjects  "
              f"cols: {list(df_tmp.columns[:6])} ...")
    except Exception as exc:
        print(f"  Tract {tract_id:6d} — could not load: {exc}")

if not dia_obj_frames:
    raise RuntimeError(
        "No dia_object data could be loaded. "
        "Check the collection name and tract IDs."
    )

df_dia_obj_all = pd.concat(dia_obj_frames, ignore_index=True)

# Remove duplicates that can arise at tract boundaries
if 'diaObjectId' in df_dia_obj_all.columns:
    n_before = len(df_dia_obj_all)
    df_dia_obj_all = df_dia_obj_all.drop_duplicates(subset='diaObjectId')
    print(f"\nDrop duplicates: {n_before:,} → {len(df_dia_obj_all):,}")

print(f"Total DiaObjects loaded (all tracts): {len(df_dia_obj_all):,}")
print(f"All columns ({len(df_dia_obj_all.columns)}):")
print(list(df_dia_obj_all.columns))

In [ ]:
# -------------------------------------------------------------------------
# Auto-detect RA/Dec and nDiaSources column names
# (names vary across stack versions)
# -------------------------------------------------------------------------
RA_COL, DEC_COL = None, None
for ra_c, dec_c in [('ra', 'dec'), ('coord_ra', 'coord_dec'),
                     ('RA', 'DEC'), ('ra_deg', 'dec_deg')]:
    if ra_c in df_dia_obj_all.columns and dec_c in df_dia_obj_all.columns:
        RA_COL, DEC_COL = ra_c, dec_c
        break

if RA_COL is None:
    raise RuntimeError(
        "Cannot find RA/Dec columns.\n"
        f"Available columns: {list(df_dia_obj_all.columns)}"
    )

NSRC_COL = next(
    (c for c in ['nDiaSources', 'n_dia_sources', 'ndiasources']
     if c in df_dia_obj_all.columns), None
)

print(f"RA column         : '{RA_COL}'")
print(f"Dec column        : '{DEC_COL}'")
print(f"nDiaSources column: '{NSRC_COL}'")

In [ ]:
# -------------------------------------------------------------------------
# Spatial cone cut: keep only objects within CONE_RADIUS_DEG of the DDF center.
# The Butler loads the full tract; the spatial cut must be applied in Python.
# -------------------------------------------------------------------------
center_sky = SkyCoord(ra=RA_CENTER * u.deg, dec=DEC_CENTER * u.deg)
obj_sky    = SkyCoord(
    ra=df_dia_obj_all[RA_COL].values   * u.deg,
    dec=df_dia_obj_all[DEC_COL].values * u.deg,
)
sep_deg = center_sky.separation(obj_sky).deg

mask_cone       = sep_deg <= CONE_RADIUS_DEG
df_dia_obj_cone = df_dia_obj_all[mask_cone].copy()
df_dia_obj_cone['_sep_deg'] = sep_deg[mask_cone]

print(f"DiaObjects in cone ({CONE_RADIUS_DEG}°): {len(df_dia_obj_cone):,}  "
      f"(from {len(df_dia_obj_all):,} total in tracts)")

In [ ]:
# -------------------------------------------------------------------------
# Filter: keep only DiaObjects with nDiaSources >= MIN_NSOURCES
# -------------------------------------------------------------------------
if NSRC_COL is not None:
    df_dia_obj_rich = (
        df_dia_obj_cone[df_dia_obj_cone[NSRC_COL] >= MIN_NSOURCES]
        .sort_values(NSRC_COL, ascending=False)
        .reset_index(drop=True)
    )
else:
    print("WARNING: nDiaSources column not found — keeping all objects in cone.")
    df_dia_obj_rich = df_dia_obj_cone.copy()

print(f"DiaObjects with {NSRC_COL} >= {MIN_NSOURCES}: {len(df_dia_obj_rich):,}")
if NSRC_COL and len(df_dia_obj_cone) > 0:
    frac = len(df_dia_obj_rich) / len(df_dia_obj_cone) * 100
    print(f"  Fraction of cone sample : {frac:.2f}%")
    if len(df_dia_obj_rich) > 0:
        print(f"  Max  {NSRC_COL} : {df_dia_obj_rich[NSRC_COL].max()}")
        print(f"  Median {NSRC_COL}: {df_dia_obj_rich[NSRC_COL].median():.1f}")

In [ ]:
# Display the top objects
preview_cols = ['diaObjectId', RA_COL, DEC_COL, NSRC_COL, '_tract', '_sep_deg']
preview_cols = [c for c in preview_cols if c in df_dia_obj_rich.columns]
print(f"Top 20 DiaObjects (sorted by {NSRC_COL}):")
display(df_dia_obj_rich[preview_cols].head(20))

---
## 6. Diagnostic Plots

In [ ]:
# =========================================================================
# Figure 1 — nDiaSources distribution: full cone vs filtered sample
# =========================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full cone (log y-scale)
ax = axes[0]
if NSRC_COL and len(df_dia_obj_cone) > 0:
    ax.hist(df_dia_obj_cone[NSRC_COL].dropna(), bins=100, log=True,
            color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(MIN_NSOURCES, color='red', linestyle='--', linewidth=1.5,
               label=f'Cut: {NSRC_COL} ≥ {MIN_NSOURCES}')
ax.set_xlabel(f'{NSRC_COL} (all bands)', fontsize=11)
ax.set_ylabel('Number of DiaObjects', fontsize=11)
ax.set_title(
    f'{SELECTED_DDF} — All objects in cone\n(N = {len(df_dia_obj_cone):,})',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: filtered sample
ax = axes[1]
if NSRC_COL and len(df_dia_obj_rich) > 0:
    ax.hist(df_dia_obj_rich[NSRC_COL].dropna(), bins=50,
            color='darkorange', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(
        f'{SELECTED_DDF} — Filtered ({NSRC_COL} ≥ {MIN_NSOURCES})\n'
        f'(N = {len(df_dia_obj_rich):,})',
        fontsize=11, fontweight='bold'
    )
else:
    ax.text(0.5, 0.5, 'No objects pass the cut',
            transform=ax.transAxes, ha='center', fontsize=13)
ax.set_xlabel(f'{NSRC_COL}', fontsize=11)
ax.set_ylabel('Number of DiaObjects', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'DiaObj_nSources_hist_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =========================================================================
# Figure 2 — Sky map: all objects (grey) + filtered objects (colored by nDiaSources)
# =========================================================================
BAND_COLORS = {'u': 'purple', 'g': 'green', 'r': 'red',
               'i': 'darkorange', 'z': 'sienna', 'y': 'black'}

fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor('#f5f5f5')
ax.invert_xaxis()   # RA increases towards the left by convention

# All objects in cone (faint background)
ax.scatter(
    df_dia_obj_cone[RA_COL], df_dia_obj_cone[DEC_COL],
    s=1, color='lightgrey', alpha=0.4, zorder=1,
    label=f'All in cone ({len(df_dia_obj_cone):,})'
)

# Filtered objects — color by nDiaSources
if NSRC_COL and len(df_dia_obj_rich) > 0:
    sc = ax.scatter(
        df_dia_obj_rich[RA_COL], df_dia_obj_rich[DEC_COL],
        c=df_dia_obj_rich[NSRC_COL],
        cmap='plasma', s=14, alpha=0.9, zorder=3,
        norm=mcolors.LogNorm(
            vmin=df_dia_obj_rich[NSRC_COL].min(),
            vmax=df_dia_obj_rich[NSRC_COL].max()),
        label=f'{NSRC_COL} ≥ {MIN_NSOURCES} ({len(df_dia_obj_rich):,})'
    )
    cbar = plt.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label(f'{NSRC_COL} (log scale)', fontsize=10)

# DDF center marker
ax.plot(RA_CENTER, DEC_CENTER,
        marker='*', markersize=18,
        color='gold', markeredgecolor='black', markeredgewidth=1.0,
        zorder=10, linestyle='none',
        label=f'{SELECTED_DDF} center')

# Approximate cone boundary
cos_dec = np.cos(np.deg2rad(DEC_CENTER))
theta   = np.linspace(0, 2 * np.pi, 361)
ax.plot(
    RA_CENTER + CONE_RADIUS_DEG / cos_dec * np.cos(theta),
    DEC_CENTER + CONE_RADIUS_DEG * np.sin(theta),
    color='red', linewidth=1.2, linestyle='--', zorder=4,
    label=f'Cone {CONE_RADIUS_DEG}°'
)

ax.set_xlabel('Right Ascension (deg)', fontsize=12)
ax.set_ylabel('Declination (deg)', fontsize=12)
ax.set_title(
    f'DP2 DiaObjects — {SELECTED_DDF}\n'
    f'(Butler, tracts {tract_ids})',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(f'DiaObj_skymap_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Load DiaSources for the Filtered DiaObjects

The `dia_source` dataset shares the same `{skymap, tract}` dimensions as
`dia_object`.  We load it per tract and then keep only rows whose
`diaObjectId` is in the filtered DiaObject set.

In [ ]:
# -------------------------------------------------------------------------
# Load dia_source per tract; keep only rows linked to filtered DiaObjects
# -------------------------------------------------------------------------
rich_object_ids = set(df_dia_obj_rich['diaObjectId'].values)

dia_src_frames = []
for tract_id in tract_ids:
    dataId = {"skymap": SKYMAP_NAME, "tract": tract_id}
    refs = list(
        registry.queryDatasets(
            "dia_source",
            collections=COLLECTION,
            dataId=dataId,
        )
    )
    for ref in refs:
        try:
            obj    = butler.get(ref)
            df_tmp = to_dataframe(obj)
            if 'diaObjectId' in df_tmp.columns:
                df_tmp = df_tmp[df_tmp['diaObjectId'].isin(rich_object_ids)]
            dia_src_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} DiaSources (filtered)")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — could not load dia_source: {exc}")

if dia_src_frames:
    df_dia_src_rich = pd.concat(dia_src_frames, ignore_index=True)
    if 'diaSourceId' in df_dia_src_rich.columns:
        df_dia_src_rich = df_dia_src_rich.drop_duplicates(subset='diaSourceId')
    print(f"\nTotal DiaSources for filtered DiaObjects: {len(df_dia_src_rich):,}")
    print(f"Columns: {list(df_dia_src_rich.columns[:15])} ...")
else:
    print("No DiaSources loaded.")
    df_dia_src_rich = pd.DataFrame()

In [ ]:
# Auto-detect column names in the DiaSource table
MJD_COL  = next((c for c in ['midPointMjdTai', 'midpointMjdTai',
                               'midpoint_mjd_tai', 'mjd']
                  if c in df_dia_src_rich.columns), None)
BAND_COL = next((c for c in ['band', 'filterName', 'filter']
                  if c in df_dia_src_rich.columns), None)
FLUX_COL = next((c for c in ['psfFlux', 'psf_flux', 'flux']
                  if c in df_dia_src_rich.columns), None)
FERR_COL = next((c for c in ['psfFluxErr', 'psf_flux_err']
                  if c in df_dia_src_rich.columns), None)

print(f"DiaSource MJD  : {MJD_COL}")
print(f"DiaSource band : {BAND_COL}")
print(f"DiaSource flux : {FLUX_COL} ± {FERR_COL}")

In [ ]:
# =========================================================================
# Figure 3 — Flux vs MJD scatter for all DiaSources of filtered DiaObjects
# =========================================================================
if MJD_COL and FLUX_COL and BAND_COL and len(df_dia_src_rich) > 0:
    fig, ax = plt.subplots(figsize=(13, 5))
    for band, grp in df_dia_src_rich.groupby(BAND_COL):
        ax.scatter(
            grp[MJD_COL], grp[FLUX_COL],
            s=1, alpha=0.2,
            color=BAND_COLORS.get(str(band), 'grey'),
            label=str(band)
        )
    ax.set_xlabel('MJD (TAI)', fontsize=12)
    ax.set_ylabel('PSF flux (nJy)', fontsize=12)
    ax.set_title(
        f'All DiaSources — {SELECTED_DDF}  '
        f'({NSRC_COL} ≥ {MIN_NSOURCES},  N_src = {len(df_dia_src_rich):,})',
        fontsize=12, fontweight='bold'
    )
    ax.legend(title='Band', fontsize=9, markerscale=6)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'DiaObj_sources_scatter_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Insufficient column information for MJD-flux scatter plot.")

---
## 8. Load Forced Photometry (ForcedSourceOnDiaObject)

Forced photometry provides a PSF flux measurement at the DiaObject position
in **every visit image**, not only those with an above-threshold detection.
This is the recommended product for light curve analysis.

In [ ]:
# -------------------------------------------------------------------------
# Load forced-source dataset per tract; keep only rows for filtered DiaObjects
# -------------------------------------------------------------------------
df_forced_rich = pd.DataFrame()

if FORCED_ON_DIA_TYPE is None:
    print("No forced-source dataset type found in the collection — skipping.")
else:
    forced_frames = []
    for tract_id in tract_ids:
        dataId = {"skymap": SKYMAP_NAME, "tract": tract_id}
        refs   = list(
            registry.queryDatasets(
                FORCED_ON_DIA_TYPE,
                collections=COLLECTION,
                dataId=dataId,
            )
        )
        for ref in refs:
            try:
                obj    = butler.get(ref)
                df_tmp = to_dataframe(obj)
                if 'diaObjectId' in df_tmp.columns:
                    df_tmp = df_tmp[df_tmp['diaObjectId'].isin(rich_object_ids)]
                forced_frames.append(df_tmp)
                print(f"  Tract {tract_id:6d} — {len(df_tmp):,} forced rows (filtered)")
            except Exception as exc:
                print(f"  Tract {tract_id:6d} — could not load {FORCED_ON_DIA_TYPE}: {exc}")

    if forced_frames:
        df_forced_rich = pd.concat(forced_frames, ignore_index=True)
        print(f"\nTotal forced-source rows: {len(df_forced_rich):,}")
        print(f"Columns: {list(df_forced_rich.columns[:15])} ...")
    else:
        print("No forced photometry rows loaded.")

In [ ]:
# Auto-detect column names in the forced-source table
FMJD_COL  = next((c for c in ['midpointMjdTai', 'midPointMjdTai',
                                'midpoint_mjd_tai', 'mjd']
                   if c in df_forced_rich.columns), None)
FBAND_COL = next((c for c in ['band', 'filterName', 'filter']
                   if c in df_forced_rich.columns), None)
FFLUX_COL = next((c for c in ['psfFlux', 'psf_flux', 'flux']
                   if c in df_forced_rich.columns), None)
FFERR_COL = next((c for c in ['psfFluxErr', 'psf_flux_err']
                   if c in df_forced_rich.columns), None)
FDIFF_COL = next((c for c in ['psfDiffFlux', 'psf_diff_flux', 'diffFlux']
                   if c in df_forced_rich.columns), None)
FDERR_COL = next((c for c in ['psfDiffFluxErr', 'psf_diff_flux_err']
                   if c in df_forced_rich.columns), None)

print(f"Forced MJD    : {FMJD_COL}")
print(f"Forced band   : {FBAND_COL}")
print(f"Forced flux   : {FFLUX_COL} ± {FFERR_COL}")
print(f"Diff flux     : {FDIFF_COL} ± {FDERR_COL}")

In [ ]:
# =========================================================================
# Figure 4 — Forced-photometry light curve of the most-detected DiaObject
# =========================================================================
if (
    len(df_forced_rich) > 0
    and len(df_dia_obj_rich) > 0
    and FMJD_COL and FBAND_COL and FFLUX_COL
):
    top_id = df_dia_obj_rich.iloc[0]['diaObjectId']
    top_n  = df_dia_obj_rich.iloc[0][NSRC_COL] if NSRC_COL else 'N/A'
    df_lc  = df_forced_rich[df_forced_rich['diaObjectId'] == top_id].sort_values(FMJD_COL)

    nrows = 2 if (FDIFF_COL and FDIFF_COL in df_lc.columns) else 1
    fig, axes = plt.subplots(nrows, 1, figsize=(13, 5 * nrows), sharex=True)
    if nrows == 1:
        axes = [axes]

    # Top panel: science image forced flux
    ax = axes[0]
    for band, grp in df_lc.groupby(FBAND_COL):
        yerr = grp[FFERR_COL] if FFERR_COL else None
        ax.errorbar(
            grp[FMJD_COL], grp[FFLUX_COL], yerr=yerr,
            fmt='o', ms=4, alpha=0.8, capsize=2,
            color=BAND_COLORS.get(str(band), 'grey'),
            label=str(band)
        )
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_ylabel('Science PSF flux (nJy)', fontsize=11)
    ax.set_title(
        f'Forced-photometry light curve — diaObjectId = {top_id}\n'
        f'{SELECTED_DDF}  |  {NSRC_COL} = {top_n}',
        fontsize=12, fontweight='bold'
    )
    ax.legend(title='Band', fontsize=9)
    ax.grid(True, alpha=0.3)

    # Bottom panel: difference image flux (if available)
    if nrows == 2:
        ax = axes[1]
        for band, grp in df_lc.groupby(FBAND_COL):
            yerr = grp[FDERR_COL] if FDERR_COL else None
            ax.errorbar(
                grp[FMJD_COL], grp[FDIFF_COL], yerr=yerr,
                fmt='o', ms=4, alpha=0.8, capsize=2,
                color=BAND_COLORS.get(str(band), 'grey'),
                label=str(band)
            )
        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_ylabel('Difference image PSF flux (nJy)', fontsize=11)
        ax.legend(title='Band', fontsize=9)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('MJD (TAI)', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'DiaObj_forcedLC_{top_id}_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Insufficient data for forced-photometry light curve plot.")

---
## 9. Coordinate-Based Cross-Match with a Fink Alert

### Context

The **DRP** (`diaObjectId` in this notebook) and the **Rubin alert broker**
pipelines (whose `diaObjectId` Fink reports) are **independent**.  There is
no reliable mapping between their IDs.

The correct workflow is:
1. Provide the (RA, Dec) of the Fink alert.
2. Compute angular separation to every DiaObject in `df_dia_obj_cone`.
3. Rank candidates by separation and apply the matching radius cut.
4. Retrieve the forced-photometry light curve of the best DRP candidate.

In [ ]:
# =========================================================================
# User-supplied Fink alert position — edit here
# =========================================================================

# Replace with the (RA, Dec) from the Fink alert broker (degrees, ICRS J2000)
FINK_RA  = 53.125    # degrees — example inside ECDFS
FINK_DEC = -28.100   # degrees

# Maximum acceptable angular separation for a valid cross-match (arcseconds)
XMATCH_RADIUS_ARCSEC = 1.0

print(f"Fink alert position : RA = {FINK_RA:.6f}°,  Dec = {FINK_DEC:+.6f}°")
print(f"Cross-match radius  : {XMATCH_RADIUS_ARCSEC} arcsec")

In [ ]:
# -------------------------------------------------------------------------
# Compute angular separations from the Fink alert to all DiaObjects in cone
# -------------------------------------------------------------------------
alert_sky  = SkyCoord(ra=FINK_RA * u.deg, dec=FINK_DEC * u.deg)
all_sky    = SkyCoord(
    ra=df_dia_obj_cone[RA_COL].values   * u.deg,
    dec=df_dia_obj_cone[DEC_COL].values * u.deg,
)
sep_arcsec = alert_sky.separation(all_sky).arcsec

df_xmatch = df_dia_obj_cone.copy()
df_xmatch['_xmatch_sep_arcsec'] = sep_arcsec

# Apply the strict matching radius
df_xmatch_cut = (
    df_xmatch[df_xmatch['_xmatch_sep_arcsec'] <= XMATCH_RADIUS_ARCSEC]
    .sort_values('_xmatch_sep_arcsec')
    .reset_index(drop=True)
)

print(f"Candidates within {XMATCH_RADIUS_ARCSEC}\" of the Fink alert: {len(df_xmatch_cut)}")

if len(df_xmatch_cut) == 0:
    print(
        "No DRP DiaObject found within the matching radius.\n"
        "Suggestions:\n"
        "  1. Increase XMATCH_RADIUS_ARCSEC (try 2 or 3 arcsec).\n"
        "  2. Verify the Fink alert coordinates.\n"
        "  3. The DRP may not have detected this object (pipeline differences)."
    )
else:
    show_cols = ['diaObjectId', RA_COL, DEC_COL, '_xmatch_sep_arcsec']
    if NSRC_COL:
        show_cols.append(NSRC_COL)
    show_cols = [c for c in show_cols if c in df_xmatch_cut.columns]
    print("Cross-match candidates:")
    display(df_xmatch_cut[show_cols].head(10))

    best_drp_id = df_xmatch_cut.iloc[0]['diaObjectId']
    best_sep    = df_xmatch_cut.iloc[0]['_xmatch_sep_arcsec']
    best_ra     = df_xmatch_cut.iloc[0][RA_COL]
    best_dec    = df_xmatch_cut.iloc[0][DEC_COL]
    print(f"\nBest DRP DiaObject ID : {best_drp_id}")
    print(f"Angular separation    : {best_sep:.4f} arcsec")
    print(f"DRP position          : RA={best_ra:.6f}°  Dec={best_dec:+.6f}°")

In [ ]:
# -------------------------------------------------------------------------
# Retrieve the light curve of the best cross-match candidate
# (from forced photometry, or fall back to DiaSources)
# -------------------------------------------------------------------------
if len(df_xmatch_cut) == 0:
    print("No cross-match — light curve plot skipped.")
else:
    best_drp_id = df_xmatch_cut.iloc[0]['diaObjectId']
    best_sep    = df_xmatch_cut.iloc[0]['_xmatch_sep_arcsec']

    # Prefer forced photometry; fall back to DiaSources
    df_best_lc = pd.DataFrame()
    lc_source  = None

    if len(df_forced_rich) > 0 and FMJD_COL:
        df_best_lc = df_forced_rich[
            df_forced_rich['diaObjectId'] == best_drp_id
        ].sort_values(FMJD_COL).copy()
        lc_source = "ForcedSourceOnDiaObject"
        _mjd_col, _band_col = FMJD_COL, FBAND_COL
        _flux_col, _ferr_col = FFLUX_COL, FFERR_COL
        _diff_col, _derr_col = FDIFF_COL, FDERR_COL

    if len(df_best_lc) == 0 and len(df_dia_src_rich) > 0 and MJD_COL:
        df_best_lc = df_dia_src_rich[
            df_dia_src_rich['diaObjectId'] == best_drp_id
        ].sort_values(MJD_COL).copy()
        lc_source  = "DiaSource"
        _mjd_col, _band_col = MJD_COL, BAND_COL
        _flux_col, _ferr_col = FLUX_COL, FERR_COL
        _diff_col, _derr_col = None, None

    if len(df_best_lc) == 0:
        print(f"No light curve data found for diaObjectId = {best_drp_id}.")
    else:
        print(f"Light curve source : {lc_source}  ({len(df_best_lc)} epochs)")

        nrows = 2 if (_diff_col and _diff_col in df_best_lc.columns) else 1
        fig, axes = plt.subplots(nrows, 1, figsize=(13, 5 * nrows), sharex=True)
        if nrows == 1:
            axes = [axes]

        ax = axes[0]
        for band, grp in df_best_lc.groupby(_band_col):
            yerr = grp[_ferr_col] if _ferr_col and _ferr_col in grp else None
            ax.errorbar(
                grp[_mjd_col], grp[_flux_col], yerr=yerr,
                fmt='o', ms=4, alpha=0.85, capsize=2,
                color=BAND_COLORS.get(str(band), 'grey'),
                label=str(band)
            )
        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_ylabel('PSF flux (nJy)', fontsize=11)
        ax.set_title(
            f'DRP light curve ({lc_source}) — cross-matched DiaObject\n'
            f'diaObjectId = {best_drp_id}   sep = {best_sep:.3f}"\n'
            f'Fink alert: RA={FINK_RA:.5f}°  Dec={FINK_DEC:+.5f}°',
            fontsize=11, fontweight='bold'
        )
        ax.legend(title='Band', fontsize=9)
        ax.grid(True, alpha=0.3)

        if nrows == 2:
            ax = axes[1]
            for band, grp in df_best_lc.groupby(_band_col):
                yerr = grp[_derr_col] if _derr_col and _derr_col in grp else None
                ax.errorbar(
                    grp[_mjd_col], grp[_diff_col], yerr=yerr,
                    fmt='o', ms=4, alpha=0.85, capsize=2,
                    color=BAND_COLORS.get(str(band), 'grey'),
                    label=str(band)
                )
            ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
            ax.set_ylabel('Difference image PSF flux (nJy)', fontsize=11)
            ax.legend(title='Band', fontsize=9)
            ax.grid(True, alpha=0.3)

        axes[-1].set_xlabel('MJD (TAI)', fontsize=11)
        plt.tight_layout()
        plt.savefig(
            f'DiaObj_xmatch_LC_{best_drp_id}_{SELECTED_DDF}.png',
            dpi=150, bbox_inches='tight'
        )
        plt.show()

---
## 10. Save Results

In [ ]:
# -------------------------------------------------------------------------
# Export filtered catalogs to CSV for downstream analysis
# -------------------------------------------------------------------------

# Filtered DiaObject table
out_obj = f"DiaObjects_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
df_dia_obj_rich.to_csv(out_obj, index=False)
print(f"Saved DiaObject table         : {out_obj}  ({len(df_dia_obj_rich):,} rows)")

# DiaSources for filtered DiaObjects
if len(df_dia_src_rich) > 0:
    out_src = f"DiaSources_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
    df_dia_src_rich.to_csv(out_src, index=False)
    print(f"Saved DiaSource table         : {out_src}  ({len(df_dia_src_rich):,} rows)")

# Forced photometry for filtered DiaObjects
if len(df_forced_rich) > 0:
    out_forced = f"ForcedSrc_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"
    df_forced_rich.to_csv(out_forced, index=False)
    print(f"Saved ForcedSource table      : {out_forced}  ({len(df_forced_rich):,} rows)")

---
## Technical Summary

| Parameter | Value |
|---|---|
| Butler repository | `dp2_prep` |
| Collection | `LSSTCam/runs/DRP/v30_0_4/DM-54249` |
| SkyMap | `lsst_cells_v2` |
| Instrument | `LSSTCam` |
| Selected DDF | configurable via `SELECTED_DDF` |
| Cone radius | configurable via `CONE_RADIUS_DEG` |
| Min nDiaSources | configurable via `MIN_NSOURCES` |
| Cross-match radius | configurable via `XMATCH_RADIUS_ARCSEC` |

### Design notes

- **No TAP** — all catalog access is through `butler.registry.queryDatasets` +
  `butler.get`, which is the correct approach at USDF before a TAP service is
  deployed for DP2.
- **Dimension introspection** (Section 3) discovers the correct `dataId` keys at
  runtime, making the notebook forward-compatible with schema changes between
  DRP processing runs.
- **Column name auto-detection** (Sections 5, 7, 8) handles the fact that
  column names vary across stack versions (e.g. `midPointMjdTai` vs
  `midpointMjdTai`).
- **Why coordinate cross-match?** The Alert Production pipeline and the DRP
  assign different `diaObjectId` values to the same physical source.  Angular
  separation is the only reliable cross-identification method between
  Fink/ALeRCE alerts and DRP catalogs.